# Patient Re-admission Indicator

## 1) Problem Statement

- This project helps understand what elements play a role in whether or not a patient will be redmitted into a hospital.

## 2) Data Collection

- Dataset Source - https://archive.ics.uci.edu/dataset/296/diabetes+130-us+hospitals+for+years+1999-2008

In [63]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
%matplotlib inline
import warnings
warnings.filterwarnings('ignore')

In [64]:
diabetes = pd.read_csv('data/diabetic_data.csv')
diabetes.head() 

,encounter_id,patient_nbr,race,gender,age,weight,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,...,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted
0,2278392,8222157,Caucasian,Female,[0-10),?,6,25,1,1,...,No,No,No,No,No,No,No,No,No,NO
1,149190,55629189,Caucasian,Female,[10-20),?,1,1,7,3,...,No,Up,No,No,No,No,No,Ch,Yes,>30
2,64410,86047875,AfricanAmerican,Female,[20-30),?,1,1,7,2,...,No,No,No,No,No,No,No,No,Yes,NO
3,500364,82442376,Caucasian,Male,[30-40),?,1,1,7,2,...,No,Up,No,No,No,No,No,Ch,Yes,NO
4,16680,42519267,Caucasian,Male,[40-50),?,1,1,7,1,...,No,Steady,No,No,No,No,No,Ch,Yes,NO


In [65]:
df = pd.read_csv('data/IDS_mapping.csv')
df.head() 

,admission_type_id,description
0,1,Emergency
1,2,Urgent
2,3,Elective
3,4,Newborn
4,5,Not Available


Descriptions are too long so it'll be better to keep them as numbers in our main dataframe instead of merging them.

### Shape of the Dataset

In [66]:
diabetes.shape

(101766, 50)

### Columns of the Dataset

In [67]:
diabetes.columns

Index(['encounter_id', 'patient_nbr', 'race', 'gender', 'age', 'weight',
       'admission_type_id', 'discharge_disposition_id', 'admission_source_id',
       'time_in_hospital', 'payer_code', 'medical_specialty',
       'num_lab_procedures', 'num_procedures', 'num_medications',
       'number_outpatient', 'number_emergency', 'number_inpatient', 'diag_1',
       'diag_2', 'diag_3', 'number_diagnoses', 'max_glu_serum', 'A1Cresult',
       'metformin', 'repaglinide', 'nateglinide', 'chlorpropamide',
       'glimepiride', 'acetohexamide', 'glipizide', 'glyburide', 'tolbutamide',
       'pioglitazone', 'rosiglitazone', 'acarbose', 'miglitol', 'troglitazone',
       'tolazamide', 'examide', 'citoglipton', 'insulin',
       'glyburide-metformin', 'glipizide-metformin',
       'glimepiride-pioglitazone', 'metformin-rosiglitazone',
       'metformin-pioglitazone', 'change', 'diabetesMed', 'readmitted'],
      dtype='object')

### We will start mapping

In [68]:
admission_type_id = { 1 : 'Emergency'
, 2 : 'Urgent'
, 3 : 'Elective'
, 4 : 'Newborn'
, 5 : 'Not Available'
, 6 : 'NULL'
, 7 : 'Trauma Center'
, 8 : 'Not Mapped' }

In [69]:
discharge_disposition_id = { 1 : 'Discharged to home'
, 2 : 'Discharged/transferred to another short term hospital'
, 3 : 'Discharged/transferred to SNF'
, 4 : 'Discharged/transferred to ICF'
, 5 : 'Discharged/transferred to another type of inpatient care institution'
, 6 : 'Discharged/transferred to home with home health service'
, 7 : 'Left AMA'
, 8 : 'Discharged/transferred to home under care of Home IV provider'
, 9 : 'Admitted as an inpatient to this hospital'
, 10 : 'Neonate discharged to another hospital for neonatal aftercare'
, 11 : 'Expired'
, 12 : 'Still patient or expected to return for outpatient services'
, 13 : 'Hospice / home'
, 14 : 'Hospice / medical facility'
, 15 : 'Discharged/transferred within this institution to Medicare approved swing bed'
, 16 : 'Discharged/transferred/referred another institution for outpatient services'
, 17 : 'Discharged/transferred/referred to this institution for outpatient services'
, 18 : 'NULL'
, 19 : 'Expired at home. Medicaid only, hospice'
, 20 : 'Expired in a medical facility. Medicaid only, hospice'
, 21 : 'Expired, place unknown. Medicaid only, hospice'
, 22 : 'Discharged/transferred to another rehab fac including rehab units of a hospital'
, 23 : 'Discharged/transferred to a long term care hospital'
, 24 : 'Discharged/transferred to a nursing facility certified under Medicaid but not certified under Medicare'
, 25 : 'Not Mapped'
, 26 : 'Unknown/Invalid'
, 30 : 'Discharged/transferred to another Type of Health Care Institution not Defined Elsewhere'
, 27 : 'Discharged/transferred to a federal health care facility'
, 28 : 'Discharged/transferred/referred to a psychiatric hospital of psychiatric distinct part unit of a hospital'
, 29 : 'Discharged/transferred to a Critical Access Hospital (CAH)' }

In [70]:
admission_source_id = { 1 : 'Physician Referral'
, 2 : 'Clinic Referral'
, 3 : 'HMO Referral'
, 4 : 'Transfer from a hospital'
, 5 : 'Transfer from a Skilled Nursing Facility (SNF)'
, 6 : 'Transfer from another health care facility'
, 7 : 'Emergency Room'
, 8 : 'Court/Law Enforcement'
, 9 :  'Not Available'
, 10 : 'Transfer from critial access hospital'
, 11 : 'Normal Delivery'
, 12 : 'Premature Delivery'
, 13 : 'Sick Baby'
, 14 : 'Extramural Birth'
, 15 : 'Not Available'
, 17 : 'NULL'
, 18 : 'Transfer From Another Home Health Agency'
, 19 : 'Readmission to Same Home Health Agency'
, 20 : 'Not Mapped'
, 21 : 'Unknown/Invalid'
, 22 : 'Transfer from hospital inpt/same fac reslt in a sep claim'
, 23 : 'Born inside this hospital'
, 24 : 'Born outside this hospital'
, 25 : 'Transfer from Ambulatory Surgery Center'
, 26 : 'Transfer from Hospice'}

In [71]:
diabetes['expiration_ind'] = diabetes['discharge_disposition_id'].isin([11,13,14,19,20,21]).astype('int')


We will install values from lookup dictionaries

In [72]:
diabetes['admission_type'] = diabetes['admission_type_id'].map(admission_type_id)
diabetes['discharge_disposition'] = diabetes['discharge_disposition_id'].map(discharge_disposition_id)
diabetes['admission_source'] = diabetes['admission_source_id'].map(admission_source_id)

diabetes['MP_DM_payer_ind'] = ((diabetes['payer_code'] == 'MP') | 
                               (diabetes['payer_code'] == 'DM')).astype(int)

del admission_type_id
del discharge_disposition_id 
del admission_source_id

diabetes['admission_grp_1_ind'] = ( diabetes['admission_type'].isin(['NULL','Emergency'])).astype(int)
diabetes['admission_grp_2_ind'] = ( diabetes['admission_type'].isin(['Elective','Not Mapped'])).astype(int)

diabetes['discharge_grp_1_ind'] = ( diabetes['discharge_disposition'].isin(['Discharged/transferred to a long term care hospital'
                                                                           ,'NULL'
                                                                           ,'Discharged to home'])).astype(int)

diabetes['discharge_grp_2_ind'] = ( diabetes['discharge_disposition'].isin(['Left AMA'
                                                                            ,'Discharged/transferred to another type of inpatient care institution'
                                                                            ,'Discharged/transferred to SNF'
                                                                            ,'Discharged/transferred to home with home health service'
                                                                            ,'Discharged/transferred to another rehab fac including rehab units of a hospital'])).astype(int)

diabetes['admission_type_ind'] = ( diabetes['admission_source'].isin(['Clinic Referral'
                                                                     ,'Transfer from a hospital'
                                                                     ,'Transfer from another health care facility'])).astype(int)

diabetes['mb_admission_grp_1_ct'] = diabetes.groupby('patient_nbr')['admission_grp_1_ind'].transform('sum')
diabetes['mb_admission_grp_2_ct'] = diabetes.groupby('patient_nbr')['admission_grp_2_ind'].transform('sum')
diabetes['mb_discharge_grp_1_ct'] = diabetes.groupby('patient_nbr')['discharge_grp_1_ind'].transform('sum')
diabetes['mb_discharge_grp_2_ct'] = diabetes.groupby('patient_nbr')['discharge_grp_2_ind'].transform('sum')
diabetes['mb_admission_type_ct']  = diabetes.groupby('patient_nbr')['admission_type_ind'].transform('sum')

drop = []
drop.extend(['payer_code', 'admission_type_id', 'discharge_disposition_id', 'admission_source_id'])
drop.extend(['admission_grp_1_ind','admission_grp_2_ind','discharge_grp_1_ind', 'discharge_grp_2_ind','admission_type_ind'])

Clean dx code and replace missing values with ZZZ

In [73]:
diabetes['diag_1'] = diabetes['diag_1'].astype(str).str[ :3]
diabetes['diag_2'] = diabetes['diag_2'].astype(str).str[ :3]
diabetes['diag_3'] = diabetes['diag_3'].astype(str).str[ :3]

diabetes['diag_1'] = diabetes['diag_1'].replace('?', 'ZZZ')
diabetes['diag_2'] = diabetes['diag_2'].replace('?', 'ZZZ')
diabetes['diag_3'] = diabetes['diag_3'].replace('?', 'ZZZ')

**Add patient-level count of unique diagnoses codes**

We're going to melt the dataset to have al diagnoses in a single column, then group them by patient_nbr and count distinct diagnoses codes.

In [74]:
diagnosis_melted = diabetes.melt(id_vars=['patient_nbr'], value_vars=['diag_1', 'diag_2', 'diag_3'])

distinct_counts = diagnosis_melted.groupby('patient_nbr')['value'].nunique().reset_index()

distinct_counts.columns = ['patient_nbr', 'distinct_diag_count']

diabetes = diabetes.merge(distinct_counts, on='patient_nbr', how='left')

del diagnosis_melted, distinct_counts